# RF Coach Vision su Colab (GPU)

**Prima volta:**
1. Sul PC lancia `python colab/prepara_colab.py` (crea `colab/rf_coach_vision_colab.zip`).
2. Carica lo zip su Google Drive nella cartella **Il mio Drive/rf_coach_vision/** (creala se non c'è).
3. Qui su Colab: menu **Runtime → Cambia tipo di runtime → GPU T4**.
4. Esegui le celle dall'alto in basso (freccia ▶ a sinistra di ogni cella, oppure Runtime → Esegui tutte).

**Video nuovi:** caricali in *Il mio Drive/rf_coach_vision/inputs/* e scrivi il nome nella cella "Analisi". Non serve rifare lo zip.

**Risultati:** finiscono in *Il mio Drive/rf_coach_vision/outputs/* (video) e *.../pred_result/* (pallina, riusata alle esecuzioni successive).

## 1. Controllo GPU
Se qui compare un errore, il runtime non ha la GPU: vedi il punto 3 sopra.

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive
Chiede l'autorizzazione ad accedere al tuo Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Prepara il progetto
Estrae lo zip nello spazio di lavoro di Colab (più veloce di Drive) e ci copia i video e i risultati pallina già salvati su Drive.

In [ ]:
import os, shutil, zipfile, glob

DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"

zip_path = os.path.join(DRIVE_DIR, "rf_coach_vision_colab.zip")
assert os.path.exists(zip_path), f"Non trovo {zip_path}: carica lo zip in quella cartella di Drive"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(WORK_DIR)

for sub, dst in [("inputs", "inputs"), ("pred_result", "tracknet3/pred_result")]:
    src = os.path.join(DRIVE_DIR, sub)
    os.makedirs(os.path.join(WORK_DIR, dst), exist_ok=True)
    if os.path.isdir(src):
        for f in glob.glob(os.path.join(src, "*")):
            shutil.copy(f, os.path.join(WORK_DIR, dst))

print("Video disponibili:", sorted(os.listdir(os.path.join(WORK_DIR, "inputs"))))

## 4. Installa le librerie
PyTorch è già presente su Colab; servono solo queste. Circa un minuto.

In [ ]:
!pip install -q ultralytics parse

## 5. Analisi
Cambia qui il video e le opzioni, poi esegui la cella.
- `TRACKNET_MODE`: `"weight"` (preciso) o `"nonoverlap"` (veloce)
- `FORCE`: `True` ricalcola la pallina anche se è già salvata

In [ ]:
VIDEO = "zverev_djokovic_trim_swin_like.mp4"
TRACKNET_MODE = "weight"
FORCE = False

import re, time
p = os.path.join(WORK_DIR, "analyze.py")
s = open(p).read()
s = re.sub(r"^TRACKNET_MODE = .*$", f'TRACKNET_MODE = "{TRACKNET_MODE}"', s, flags=re.M)
s = re.sub(r"^TRACKNET_FORCE_RECOMPUTE = .*$", f"TRACKNET_FORCE_RECOMPUTE = {FORCE}", s, flags=re.M)
open(p, "w").write(s)

%cd {WORK_DIR}
t0 = time.time()
!python analyze.py inputs/{VIDEO}
print(f"\nTempo totale: {(time.time() - t0) / 60:.1f} minuti")

## 6. Salva i risultati su Drive
Da eseguire sempre alla fine: lo spazio di lavoro di Colab viene cancellato quando la sessione si chiude.

In [ ]:
for sub, src in [("outputs", "outputs"), ("pred_result", "tracknet3/pred_result")]:
    dst = os.path.join(DRIVE_DIR, sub)
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(os.path.join(WORK_DIR, src, "*")):
        shutil.copy(f, dst)
        print("salvato:", os.path.join(dst, os.path.basename(f)))